# Dataset 2

Name: Weather in Szeged 2006-2016

Author: Norbert Budincsevity

Purpose: The dataset was built for a project the author was conducting to compare real life historical weather to weather folklore. It includes a summary of the hourly/daily weather data in the Szegad, Hungary area between 2006 and 2016.

Shape: The dataset consists of 9,6453 rows and 12 feature columns

Features:

| Feature Name | Description | Type |
|----------|----------|----------|
| Formatted Date | Date and time of Observation |Numerical|
| Summary | Short description of weather conditions (hourly) |Categorical|
| Percip Type | Type of daily percipitation |Categorical|
| Temperature (C) | Air Temperature in degrees Celsius |Numerical|
| Apparent Temperature (C) | Perceived "feels like" air temperature |Numerical|
| Humidity | Percentage of moisture in air |Numerical|
| Wind Speed (km/h) | Current speed of wind recorded |Numerical|
| Wind Bearing (degrees) | Direction wind is blowing from |Numerical|
| Visibility (km) | Distance of clear vision |Numerical|
| Loud Cover | Level of cloud coverage recorded |Numerical|
| Pressure (millibars) | Current level of atmospheric pressure |Numerical|
| Daily Summary | General summary of daily weather |Categorical|




In [78]:
import seaborn as sns
import matplotlib.pyplot as plt
import kagglehub
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.impute import KNNImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [79]:
# dataset 

np.random.seed(42)

DATASET_PATH = kagglehub.dataset_download(
    "budincsevity/szeged-weather"
)

dataframe = pd.read_csv(f"{DATASET_PATH}/weatherHistory.csv")

print('Head:', dataframe.head())

print(len(dataframe))

Head:                   Formatted Date        Summary Precip Type  Temperature (C)  \
0  2006-04-01 00:00:00.000 +0200  Partly Cloudy        rain         9.472222   
1  2006-04-01 01:00:00.000 +0200  Partly Cloudy        rain         9.355556   
2  2006-04-01 02:00:00.000 +0200  Mostly Cloudy        rain         9.377778   
3  2006-04-01 03:00:00.000 +0200  Partly Cloudy        rain         8.288889   
4  2006-04-01 04:00:00.000 +0200  Mostly Cloudy        rain         8.755556   

   Apparent Temperature (C)  Humidity  Wind Speed (km/h)  \
0                  7.388889      0.89            14.1197   
1                  7.227778      0.86            14.2646   
2                  9.377778      0.89             3.9284   
3                  5.944444      0.83            14.1036   
4                  6.977778      0.83            11.0446   

   Wind Bearing (degrees)  Visibility (km)  Loud Cover  Pressure (millibars)  \
0                   251.0          15.8263         0.0               101

In [80]:
# check missing values

print('Missing values:', dataframe.isnull().sum())


Missing values: Formatted Date                0
Summary                       0
Precip Type                 517
Temperature (C)               0
Apparent Temperature (C)      0
Humidity                      0
Wind Speed (km/h)             0
Wind Bearing (degrees)        0
Visibility (km)               0
Loud Cover                    0
Pressure (millibars)          0
Daily Summary                 0
dtype: int64


# test 1

Imputation Method: Default Value Imputation (Median)

Description: Median Imputation is used to replace missing values of the feature 'Wind Speed (km/h)'. The median was chosen to reduce bias on imputed values that could be caused by outlying extreme wind speeds

In [81]:
# test 1
df1 = dataframe.copy()

# set 10% of indices to NaN -> MCAR
nan_indices = np.random.choice(df1.index, size=int(0.1 * len(df1)), replace=False)
df1.loc[nan_indices, 'Wind Speed (km/h)'] = np.nan

print('Missing values in Wind Speed (km/h):', df1['Wind Speed (km/h)'].isnull().sum())


Missing values in Wind Speed (km/h): 9645


In [82]:
# use default value imputation (median)

median_value = df1['Wind Speed (km/h)'].median()

df1['Wind Speed (km/h)'].fillna(median_value, inplace=True)

print('Missing values after imputation:', df1['Wind Speed (km/h)'].isnull().sum())

Missing values after imputation: 0


C:\Users\Taha Riyaan\AppData\Local\Temp\ipykernel_21612\271156055.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df1['Wind Speed (km/h)'].fillna(median_value, inplace=True)


In [83]:
# evaluation

true_values = dataframe.loc[nan_indices, 'Wind Speed (km/h)']
imputed_values = df1.loc[nan_indices, 'Wind Speed (km/h)']

mae = mean_absolute_error(true_values, imputed_values)
print(f'Mean Absolute Error of imputation: {mae:.2f} km/h')

# MAE of 5.26 km/h suggests the median is not a very effective way to imputate data

Mean Absolute Error of imputation: 5.25 km/h


# test 2

Imputation Method: Linear Regression Imputation

Description: Linear Regression imputation using the correlation between 'Temperature (C)' and 'Humidity'. Missing temperature values can be estimated based on a model trained on known humidity levels. Bivariate method.

In [84]:
df2 = dataframe.copy()

# set 10% of indices to NaN -> MCAR
nan_indices = np.random.choice(df1.index, size=int(0.1 * len(df1)), replace=False)
df2.loc[nan_indices, 'Temperature (C)'] = np.nan

print('Missing values in Temperature (C):', df1['Temperature (C)'].isnull().sum())

Missing values in Temperature (C): 0


In [85]:
# prepare linear regression
train_data = df2.dropna(subset=['Temperature (C)', 'Humidity'])
X_train = train_data[['Humidity']]
y_train = train_data['Temperature (C)']

# model
model = LinearRegression()
model.fit(X_train, y_train)

# prepare rows with missing values
missing_mask = df2['Temperature (C)'].isnull()
X_missing = df2.loc[missing_mask, ['Humidity']]

# predict and impute
df2.loc[missing_mask, 'Temperature (C)'] = model.predict(X_missing)

print('Missing values after linear regression imputation:', df2['Temperature (C)'].isnull().sum())


Missing values after linear regression imputation: 0


In [ ]:
true_values = dataframe.loc[nan_indices, 'Temperature (C)']
imputed_values = df2.loc[nan_indices, 'Temperature (C)']

mae = mean_absolute_error(true_values, imputed_values)
print(f"Mean Absolute Error of imputation: {mae:.2f} °C")

# MAE of 6.03 °C suggests that while there is some relationship between humidity and temperature, linear regression using the correlation of the two features is not
# a very effective way to impute missing temperature values in this dataset.

Mean Absolute Error (Bivariate Linear Regression): 6.07 °C


# test 3

Imputation Method: Similarity-based K-Nearest Neighbours (KNN) imputation

Description: Multivariate technique. Fills in missing 'Apparent Temperature' values by identifying K most similar rows based on 'Temperature (C)', 'Humidity', 'Visibility (km)', 'Pressure (millibars)' features.

In [ ]:
df3 = dataframe.copy()

# MAR simulation, only remove 'Apparent Temperature (°C)' values if 'Wind Speed (km/h)' > 15
wind_mask = df3['Wind Speed (km/h)'] > 15


# remove 50% of values
to_drop = df3[wind_mask].sample(frac=0.5, random_state=42).index
df3.loc[to_drop, 'Apparent Temperature (C)'] = np.nan


print(f"Missing values in Apparent Temperature (C): {df3['Apparent Temperature (C)'].isnull().sum()}")



Missing values created: 10197


In [88]:
# features that correlate with 'Apparent Temperature (C)' - 'Temperature (C)', 'Humidity', 'Visibility (km)', 'Pressure (millibars)'
features = ['Apparent Temperature (C)', 'Temperature (C)', 'Humidity', 'Visibility (km)', 'Pressure (millibars)']

# impute using KNN
imputer = KNNImputer(n_neighbors=5)
df3[features] = imputer.fit_transform(df3[features])

print('Missing values after KNN imputation:', df3['Apparent Temperature (C)'].isnull().sum())

Missing values after KNN imputation: 0


In [89]:
# evaluation

true_values = dataframe.loc[to_drop, 'Apparent Temperature (C)']
imputed_values = df3.loc[to_drop, 'Apparent Temperature (C)']

mae = mean_absolute_error(true_values, imputed_values)
print(f'Mean Absolute Error of KNN imputation: {mae:.2f} °C')

# MAE of 0.6 °C suggests the KNN imputaiton method is very effective for imputing missing apparent temperature values when using a multivariate approach.
# The approach uses the correlation between multiple features listed in Cell 3

Mean Absolute Error of KNN imputation: 0.86 °C


# References:

Budincsevity, N. (n.d.). Weather in Szeged 2006-2016. Www.kaggle.com. https://www.kaggle.com/datasets/budincsevity/szeged-weather

Waskom, M. (2012). seaborn: statistical data visualization — seaborn 0.10.1 documentation. Seaborn.pydata.org. https://seaborn.pydata.org/index.html

# Work Split